<a href="https://colab.research.google.com/github/suryanshshah2006/complete-triplet-bilstm-sleep-staging/blob/main/ZAN_EEG_EXP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install scipy scikit-learn seaborn
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import random
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
from scipy.signal import resample_poly, butter, filtfilt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import seaborn as sns
import matplotlib.pyplot as plt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SOURCE_DIR = '/content/drive/MyDrive/EEG_Full_58_Subjects'
MASTER_CSV = '/content/drive/MyDrive/EEG_Full_58_Subjects/master_labels_clean.csv'
LOCAL_DIR = '/content/local_eeg_data'
MODEL_DIR = '/content/drive/MyDrive/eeg_bilstm_complete_triplets'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
print(file_df.columns.tolist())
print(labels_df.columns.tolist())
print(master_df.columns.tolist())

['filename', 'subject', 'seg_idx', 'filepath']
['subject', 'filename', 'filepath', 'seg_idx', 'label', 'label_code', 'exists', 'epoch_rank']
['filename_x', 'subject', 'seg_idx', 'filepath_x', 'filename_y', 'filepath_y', 'label', 'label_code', 'exists', 'epoch_rank']


In [ ]:
# Fail-safe: restore local data if Colab wiped memory
if not os.path.exists(LOCAL_DIR):
    print("Unzipping calibrated data into Colab local runtime...")
    shutil.unpack_archive('/content/drive/MyDrive/Zan_EEG_128Hz_Epochs_CALIBRATED.zip', LOCAL_DIR)
    print("Local data restored!")

print("Building subject-wise epoch indexer...")

labels_df = pd.read_csv(MASTER_CSV)

files = [f for f in os.listdir(LOCAL_DIR) if f.endswith('.npy')]
file_df = pd.DataFrame({'filename': files})
file_df['subject'] = file_df['filename'].apply(lambda x: x.split('_seg_')[0])
file_df['seg_idx'] = file_df['filename'].apply(lambda x: int(x.split('_seg_')[1].replace('.npy', '')))
file_df['filepath'] = os.path.join(LOCAL_DIR, '') + file_df['filename']

master_df = pd.merge(file_df, labels_df, on=['subject', 'seg_idx'], how='inner')
master_df = master_df.sort_values(by=['subject', 'seg_idx']).reset_index(drop=True)

print("Rows:", len(master_df))
print("Subjects:", master_df['subject'].nunique())

sample_arr = np.load(master_df.iloc[0]['filepath'])
print("Sample array shape:", sample_arr.shape)

Unzipping calibrated data into Colab local runtime...
Local data restored!
Building subject-wise epoch indexer...
Rows: 47090
Subjects: 52


KeyError: 'filepath'

In [ ]:
# Fail-safe: restore local data if Colab wiped memory
if not os.path.exists(LOCAL_DIR):
    print("Unzipping calibrated data into Colab local runtime...")
    shutil.unpack_archive('/content/drive/MyDrive/Zan_EEG_128Hz_Epochs_CALIBRATED.zip', LOCAL_DIR)
    print("Local data restored!")

print("Building subject-wise epoch indexer...")

labels_df = pd.read_csv(MASTER_CSV)

# Keep only merge keys + genuine clinical info. Drop filename/filepath/epoch_rank
# since they're stale (point at old Drive paths) — file_df below rebuilds them
# fresh from the current local zip.
labels_df = labels_df[['subject', 'seg_idx', 'label', 'label_code', 'exists']]

files = [f for f in os.listdir(LOCAL_DIR) if f.endswith('.npy')]
file_df = pd.DataFrame({'filename': files})
file_df['subject'] = file_df['filename'].apply(lambda x: x.split('_seg_')[0])
file_df['seg_idx'] = file_df['filename'].apply(lambda x: int(x.split('_seg_')[1].replace('.npy', '')))
file_df['filepath'] = os.path.join(LOCAL_DIR, '') + file_df['filename']

master_df = pd.merge(file_df, labels_df, on=['subject', 'seg_idx'], how='inner')
master_df = master_df.sort_values(by=['subject', 'seg_idx']).reset_index(drop=True)

print("Rows:", len(master_df))
print("Subjects:", master_df['subject'].nunique())

sample_arr = np.load(master_df.iloc[0]['filepath'])
print("Sample array shape:", sample_arr.shape)

Building subject-wise epoch indexer...
Rows: 47090
Subjects: 52
Sample array shape: (24, 3840)


In [ ]:
master_df['epoch_rank'] = master_df.groupby('subject').cumcount()

# Epoch-level split (your original approach — kept as the leakage-comparison ablation)
train_e_idx, val_e_idx = train_test_split(
    master_df.index, test_size=0.2, random_state=42, stratify=master_df['label_code']
)
master_df['split_epoch_level'] = 'train'
master_df.loc[val_e_idx, 'split_epoch_level'] = 'val'

# Subject-level split (leakage-free, primary result)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_s_idx, val_s_idx = next(gss.split(master_df, groups=master_df['subject']))
master_df['split_subject_level'] = 'train'
master_df.loc[val_s_idx, 'split_subject_level'] = 'val'

assert len(set(master_df.loc[train_s_idx,'subject']) & set(master_df.loc[val_s_idx,'subject'])) == 0

master_df.to_csv('/content/local_eeg_data/master_epoch_index.csv', index=False)
print("Epoch-level  -> train:", (master_df['split_epoch_level']=='train').sum(),
      "val:", (master_df['split_epoch_level']=='val').sum())
print("Subject-level-> train:", (master_df['split_subject_level']=='train').sum(),
      "val:", (master_df['split_subject_level']=='val').sum())

Epoch-level  -> train: 37672 val: 9418
Subject-level-> train: 36903 val: 10187


In [ ]:
lookup = master_df.set_index(['subject', 'epoch_rank'])
triplet_rows = []
for _, row in master_df.iterrows():
    sub, rank = row['subject'], int(row['epoch_rank'])
    try:
        prev_row = lookup.loc[(sub, rank - 1)]
        curr_row = lookup.loc[(sub, rank)]
        next_row = lookup.loc[(sub, rank + 1)]
        if isinstance(prev_row, pd.DataFrame): prev_row = prev_row.iloc[0]
        if isinstance(curr_row, pd.DataFrame): curr_row = curr_row.iloc[0]
        if isinstance(next_row, pd.DataFrame): next_row = next_row.iloc[0]
        if not (int(prev_row['seg_idx']) + 1 == int(curr_row['seg_idx']) and
                int(curr_row['seg_idx']) + 1 == int(next_row['seg_idx'])):
            continue
        triplet_rows.append({
            'subject': sub, 'epoch_rank': rank, 'seg_idx': int(curr_row['seg_idx']),
            'filepath_prev': prev_row['filepath'], 'filepath_curr': curr_row['filepath'],
            'filepath_next': next_row['filepath'],
            'label_code': int(curr_row['label_code']),
            'split_epoch_level': curr_row['split_epoch_level'],
            'split_subject_level': curr_row['split_subject_level'],
        })
    except KeyError:
        continue

triplet_df = pd.DataFrame(triplet_rows).sort_values(['subject', 'seg_idx']).reset_index(drop=True)
print("Complete triplet samples:", len(triplet_df))
print(triplet_df['label_code'].value_counts().sort_index())

Complete triplet samples: 46928
label_code
0     6295
1     5913
2    21933
3     5883
4     6904
Name: count, dtype: int64


In [ ]:
class CompleteTripletGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=32, shuffle=True, n_classes=5, **kwargs):
        super().__init__(**kwargs)
        self.df = df.reset_index(drop=True).copy()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min((index + 1) * self.batch_size, len(self.df))
        batch_df = self.df.iloc[self.indices[start:end]]
        X_batch, y_batch = [], []
        for _, row in batch_df.iterrows():
            X_prev = np.load(row['filepath_prev']).astype(np.float32)
            X_curr = np.load(row['filepath_curr']).astype(np.float32)
            X_next = np.load(row['filepath_next']).astype(np.float32)
            X_context = np.concatenate([X_prev, X_curr, X_next], axis=-1)
            mean = X_context.mean(axis=1, keepdims=True)
            std = X_context.std(axis=1, keepdims=True)
            X_context = (X_context - mean) / (std + 1e-8)
            X_batch.append(X_context)
            y_batch.append(row['label_code'])
        X_batch = np.array(X_batch, dtype=np.float32)
        y_batch = to_categorical(np.array(y_batch), num_classes=self.n_classes)
        return X_batch, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
def build_triplet_model():
    inp = tf.keras.Input(shape=(24, 9216))
    x = tf.keras.layers.Permute((2, 1))(inp)
    s1 = tf.keras.layers.Conv1D(32, 32, strides=4, activation='relu')(x)
    s2 = tf.keras.layers.Conv1D(32, 128, strides=4, activation='relu')(x)
    f = tf.keras.layers.Concatenate()([s1, s2])
    f = tf.keras.layers.BatchNormalization()(f)
    f = tf.keras.layers.MaxPooling1D(4)(f)
    f = tf.keras.layers.Dropout(0.3)(f)
    f = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(f)
    a = tf.keras.layers.Attention()([f, f])
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    d = tf.keras.layers.Dense(64, activation='relu')(a)
    out = tf.keras.layers.Dense(5, activation='softmax')(d)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                   loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                   metrics=['accuracy'])
    return model

In [ ]:
class SingleEpochGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=32, shuffle=True, n_classes=5, **kwargs):
        super().__init__(**kwargs)
        self.df = df.reset_index(drop=True).copy()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min((index + 1) * self.batch_size, len(self.df))
        batch_df = self.df.iloc[self.indices[start:end]]
        X_batch, y_batch = [], []
        for _, row in batch_df.iterrows():
            X = np.load(row['filepath']).astype(np.float32)
            mean = X.mean(axis=1, keepdims=True)
            std = X.std(axis=1, keepdims=True)
            X = (X - mean) / (std + 1e-8)
            X_batch.append(X)
            y_batch.append(row['label_code'])
        X_batch = np.array(X_batch, dtype=np.float32)
        y_batch = to_categorical(np.array(y_batch), num_classes=self.n_classes)
        return X_batch, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
def build_deepsleepnet(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    # small-filter branch (temporal detail)
    a = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu')(x)
    a = tf.keras.layers.MaxPooling1D(8)(a)
    a = tf.keras.layers.Dropout(0.5)(a)
    a = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(a)
    a = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(a)
    a = tf.keras.layers.MaxPooling1D(4)(a)
    # large-filter branch (frequency detail)
    b = tf.keras.layers.Conv1D(64, 400, strides=50, activation='relu')(x)
    b = tf.keras.layers.MaxPooling1D(4)(b)
    b = tf.keras.layers.Dropout(0.5)(b)
    b = tf.keras.layers.Conv1D(128, 6, activation='relu', padding='same')(b)
    b = tf.keras.layers.Conv1D(128, 6, activation='relu', padding='same')(b)
    b = tf.keras.layers.MaxPooling1D(2)(b)
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    b = tf.keras.layers.GlobalAveragePooling1D()(b)
    f = tf.keras.layers.Concatenate()([a, b])
    f = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False))(
        tf.keras.layers.Reshape((1, f.shape[-1]))(f))
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(f)
    return tf.keras.Model(inp, out)

def build_seqsleepnet(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.Conv1D(64, 32, strides=4, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=False))(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_utime(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    skips = []
    for filters in [16, 32, 64, 128]:
        x = tf.keras.layers.Conv1D(filters, 9, activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        skips.append(x)
        x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(256, 9, activation='relu', padding='same')(x)
    for filters, skip in zip([128,64,32,16], reversed(skips)):
        x = tf.keras.layers.UpSampling1D(2)(x)
        x = tf.keras.layers.Cropping1D((0, x.shape[1]-skip.shape[1]))(x) if x.shape[1] != skip.shape[1] else x
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv1D(filters, 9, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_sleepeegnet(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(8)(x)
    x = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False))(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_attnsleep(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    a = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu', padding='same')(x)
    b = tf.keras.layers.Conv1D(64, 400, strides=50, activation='relu', padding='same')(x)
    a = tf.keras.layers.MaxPooling1D(4)(a)
    b = tf.keras.layers.MaxPooling1D(2)(b)
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    b = tf.keras.layers.GlobalAveragePooling1D()(b)
    f = tf.keras.layers.Concatenate()([a, b])
    f = tf.keras.layers.Reshape((1, f.shape[-1]))(f)
    attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=32)(f, f)
    attn = tf.keras.layers.GlobalAveragePooling1D()(attn)
    x = tf.keras.layers.Dense(64, activation='relu')(attn)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_lightsleepnet(input_shape=(24,3072), n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.SeparableConv1D(32, 32, strides=4, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(64, 16, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

BASELINES = {
    'DeepSleepNet': build_deepsleepnet,
    'SeqSleepNet': build_seqsleepnet,
    'U-Time': build_utime,
    'SleepEEGNet': build_sleepeegnet,
    'AttnSleep': build_attnsleep,
    'LightSleepNet': build_lightsleepnet,
}

def compile_baseline(builder):
    m = builder()
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
               loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
               metrics=['accuracy'])
    return m

In [ ]:
def train_and_eval(gen_cls, df, split_col, model_builder, model_name, epochs=30):
    train_df = df[df[split_col] == 'train'].reset_index(drop=True)
    val_df   = df[df[split_col] == 'val'].reset_index(drop=True)
    print(f"[{model_name} | {split_col}] Train: {len(train_df)}  Val: {len(val_df)}")

    train_gen = gen_cls(train_df, batch_size=32, shuffle=True)
    val_gen   = gen_cls(val_df, batch_size=32, shuffle=False)

    model = model_builder()
    ckpt_path = os.path.join(MODEL_DIR, f"{model_name}_{split_col}.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ]
    history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks, verbose=1)

    best_model = tf.keras.models.load_model(ckpt_path)
    y_true, y_pred = [], []
    for i in range(len(val_gen)):
        Xb, yb = val_gen[i]
        preds = best_model.predict(Xb, verbose=0)
        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=['Wake','N1','N2','N3','REM'],
                                    digits=4, output_dict=True)
    print(f"[{model_name} | {split_col}] Val Accuracy: {acc:.4f}")
    return {'model': model_name, 'split': split_col, 'accuracy': acc,
            'macro_f1': report['macro avg']['f1-score'], 'history': history, 'report': report}

In [ ]:
# Confirm actual epoch length from real data instead of assuming it
_sample = np.load(master_df.iloc[0]['filepath'])
EPOCH_LEN = _sample.shape[1]
N_CHANNELS = _sample.shape[0]
print(f"Detected shape: {N_CHANNELS} channels x {EPOCH_LEN} samples")

Detected shape: 24 channels x 3840 samples


In [ ]:
# Confirmed data shape
EPOCH_LEN = 3840
N_CHANNELS = 24

def build_deepsleepnet(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    a = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu')(x)
    a = tf.keras.layers.MaxPooling1D(8)(a)
    a = tf.keras.layers.Dropout(0.5)(a)
    a = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(a)
    a = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(a)
    a = tf.keras.layers.MaxPooling1D(4)(a)
    b = tf.keras.layers.Conv1D(64, 400, strides=50, activation='relu')(x)
    b = tf.keras.layers.MaxPooling1D(4)(b)
    b = tf.keras.layers.Dropout(0.5)(b)
    b = tf.keras.layers.Conv1D(128, 6, activation='relu', padding='same')(b)
    b = tf.keras.layers.Conv1D(128, 6, activation='relu', padding='same')(b)
    b = tf.keras.layers.MaxPooling1D(2)(b)
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    b = tf.keras.layers.GlobalAveragePooling1D()(b)
    f = tf.keras.layers.Concatenate()([a, b])
    f = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False))(
        tf.keras.layers.Reshape((1, f.shape[-1]))(f))
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(f)
    return tf.keras.Model(inp, out)

def build_seqsleepnet(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.Conv1D(64, 32, strides=4, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64, return_sequences=False))(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_utime(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    skips = []
    for filters in [16, 32, 64, 128]:
        x = tf.keras.layers.Conv1D(filters, 9, activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        skips.append(x)
        x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(256, 9, activation='relu', padding='same')(x)
    for filters, skip in zip([128,64,32,16], reversed(skips)):
        x = tf.keras.layers.UpSampling1D(2)(x)
        if x.shape[1] != skip.shape[1]:
            diff = x.shape[1] - skip.shape[1]
            if diff > 0:
                x = tf.keras.layers.Cropping1D((0, diff))(x)
            else:
                skip = tf.keras.layers.Cropping1D((0, -diff))(skip)
        x = tf.keras.layers.Concatenate()([x, skip])
        x = tf.keras.layers.Conv1D(filters, 9, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_sleepeegnet(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(8)(x)
    x = tf.keras.layers.Conv1D(128, 8, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False))(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_attnsleep(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    a = tf.keras.layers.Conv1D(64, 50, strides=6, activation='relu', padding='same')(x)
    b = tf.keras.layers.Conv1D(64, 400, strides=50, activation='relu', padding='same')(x)
    a = tf.keras.layers.MaxPooling1D(4)(a)
    b = tf.keras.layers.MaxPooling1D(2)(b)
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    b = tf.keras.layers.GlobalAveragePooling1D()(b)
    f = tf.keras.layers.Concatenate()([a, b])
    f = tf.keras.layers.Reshape((1, f.shape[-1]))(f)
    attn = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=32)(f, f)
    attn = tf.keras.layers.GlobalAveragePooling1D()(attn)
    x = tf.keras.layers.Dense(64, activation='relu')(attn)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def build_lightsleepnet(input_shape, n_classes=5):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2,1))(inp)
    x = tf.keras.layers.SeparableConv1D(32, 32, strides=4, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(64, 16, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out)

BASELINES = {
    'DeepSleepNet': build_deepsleepnet,
    'SeqSleepNet': build_seqsleepnet,
    'U-Time': build_utime,
    'SleepEEGNet': build_sleepeegnet,
    'AttnSleep': build_attnsleep,
    'LightSleepNet': build_lightsleepnet,
}

def compile_baseline(builder):
    m = builder(input_shape=(N_CHANNELS, EPOCH_LEN))
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
               loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
               metrics=['accuracy'])
    return m

In [ ]:
def build_triplet_model(input_shape):
    inp = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inp)
    s1 = tf.keras.layers.Conv1D(32, 32, strides=4, activation='relu')(x)
    s2 = tf.keras.layers.Conv1D(32, 128, strides=4, activation='relu')(x)
    f = tf.keras.layers.Concatenate()([s1, s2])
    f = tf.keras.layers.BatchNormalization()(f)
    f = tf.keras.layers.MaxPooling1D(4)(f)
    f = tf.keras.layers.Dropout(0.3)(f)
    f = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(f)
    a = tf.keras.layers.Attention()([f, f])
    a = tf.keras.layers.GlobalAveragePooling1D()(a)
    d = tf.keras.layers.Dense(64, activation='relu')(a)
    out = tf.keras.layers.Dense(5, activation='softmax')(d)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                   loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                   metrics=['accuracy'])
    return model

In [ ]:
if 'results' not in globals():
    results = []

for name, builder in BASELINES.items():
    for split_col in ['split_epoch_level', 'split_subject_level']:
        results.append(
            train_and_eval(SingleEpochGenerator, master_df, split_col,
                            lambda b=builder: compile_baseline(b), name, epochs=30)
        )

[DeepSleepNet | split_epoch_level] Train: 37672  Val: 9418
Epoch 1/30
1178/1178 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.6076 - loss: 1.1060
Epoch 1: val_accuracy improved from None to 0.69909, saving model to /content/drive/MyDrive/eeg_bilstm_complete_triplets/DeepSleepNet_split_epoch_level.keras

Epoch 1: finished saving model to /content/drive/MyDrive/eeg_bilstm_complete_triplets/DeepSleepNet_split_epoch_level.keras
1178/1178 ━━━━━━━━━━━━━━━━━━━━ 206s 164ms/step - accuracy: 0.6662 - loss: 1.0053 - val_accuracy: 0.6991 - val_loss: 0.9270 - learning_rate: 5.0000e-04
Epoch 2/30
1178/1178 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.7411 - loss: 0.8782
Epoch 2: val_accuracy improved from 0.69909 to 0.77012, saving model to /content/drive/MyDrive/eeg_bilstm_complete_triplets/DeepSleepNet_split_epoch_level.keras

Epoch 2: finished saving model to /content/drive/MyDrive/eeg_bilstm_complete_triplets/DeepSleepNet_split_epoch_level.keras
1178/1178 ━━━━━━━━━━━━━━━━━━━━ 184s 157ms/s

In [ ]:
summary_df = pd.DataFrame([{
    'Model': r['model'], 'Split': r['split'],
    'Accuracy': r['accuracy'], 'Macro F1': r['macro_f1'],
    'Params': None  # fill with model.count_params() if you want it per row
} for r in results])
summary_df.to_csv(os.path.join(MODEL_DIR, 'ablation_results.csv'), index=False)
summary_df